<a href="https://colab.research.google.com/github/cerr/pycerr-notebooks/blob/main/04_registration/register_planning_ct_to_pet_ct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Register a planning CT to a PET/CT using ANTs

## Introduction

This notebook registers a **planning CT** (moving image) to the CT of a **PET/CT** study (fixed image) using Advanced Normalization Tools (ANTsPy), driven through pyCERR's `cerr.registration.ants_reg` module.

To make the intensity-based registration robust to the different fields of view of the two scans, the similarity metric is restricted to the **patient outline** on each scan (a binary mask). Once the transform is computed, an ROI (e.g. the planning GTV) is warped from the planning CT into the PET/CT frame so it can be overlaid on the PET.

## I/O

* **Inputs**: a PET/CT DICOM directory and a planning-CT DICOM directory (the latter with an associated RTSTRUCT).
* **Output**: a pyCERR `planC` with the warped planning CT and warped ROI added, plus the ANTs transform written to disk.

All paths below are **placeholders** — edit them to point at your data.

## License

By downloading the software you are agreeing to the following terms and conditions as well as to the Terms of Use of CERR software.

**`THE SOFTWARE IS PROVIDED "AS IS" AND CERR DEVELOPMENT TEAM AND ITS COLLABORATORS DO NOT MAKE ANY WARRANTY, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE, NOR DO THEY ASSUME ANY LIABILITY OR RESPONSIBILITY FOR THE USE OF THIS SOFTWARE.`**

`This software is for research purposes only and has not been approved for clinical use.`

## Install pyCERR

In [ ]:
# Install pyCERR with the ANTs registration extra (pulls in antspyx).
!pip install "pyCERR[ants] @ git+https://github.com/cerr/pyCERR.git@testing"

## Imports

In [ ]:
import os
from cerr import plan_container as pc
from cerr.registration import ants_reg
from cerr.utils import mask as maskUtils
from cerr.dataclasses import structure as cerrStr

## Define input/output paths

Replace the placeholder strings with paths to your own data. No paths are hard-coded in this notebook.

In [ ]:
# --- Placeholders: edit these ---
petCtDicomDir   = 'path/to/pet_ct_dicom_dir'      # fixed (base) study
planCtDicomDir  = 'path/to/planning_ct_dicom_dir' # moving study (+RTSTRUCT)
transformSaveDir = 'path/to/output/transforms'

os.makedirs(transformSaveDir, exist_ok=True)

## Load the scans

`loadDcmDir` appends every scan it finds to `planC`. Inspect the loaded scans and set `baseScanNum` to the **PET/CT's CT** and `movScanNum` to the **planning CT**.

In [ ]:
planC = pc.loadDcmDir(petCtDicomDir)
planC = pc.loadDcmDir(planCtDicomDir, {}, planC)

for i, s in enumerate(planC.scan):
    print(i, s.scanInfo[0].imageType, s.getScanSize())
for i, st in enumerate(planC.structure):
    print('struct', i, st.structureName)

In [ ]:
# Set these after inspecting the printout above.
baseScanNum = 0   # PET/CT CT (fixed)
movScanNum  = 1   # planning CT (moving)
gtvStrNum   = 0   # index in planC.structure of the ROI to warp (e.g. GTV)

## Build patient-outline masks

The masks restrict the ANTs similarity metric to the patient, ignoring the couch, air, and out-of-field regions that differ between the two acquisitions.

In [ ]:
baseMask3M = maskUtils.getPatientOutline(planC.scan[baseScanNum].getScanArray())
movMask3M  = maskUtils.getPatientOutline(planC.scan[movScanNum].getScanArray())

## Register

`registerScansAnts` writes the fixed/moving scans (and masks) to NIfTI, runs `ants.registration`, warps the moving scan into fixed space, and appends both the warped scan and a `Deform` object to `planC`.

`typeOfTransform` accepts any ANTs preset, e.g. `'Rigid'`, `'Affine'`, `'SyN'`, `'antsRegistrationSyNQuick[b]'` (b-spline SyN, used here).

In [ ]:
planC = ants_reg.registerScansAnts(
    planC, baseScanNum, planC, movScanNum,
    transformSaveDir=transformSaveDir,
    typeOfTransform='antsRegistrationSyNQuick[b]',
    baseMask3M=baseMask3M, movMask3M=movMask3M, maskAllStages=True)

deformS = planC.deform[-1]
print('Transform written to:', deformS.deformOutFilePath)

### Optional: landmark-based initial alignment

If the two scans start far apart, supply a few paired landmark points to seed the registration with a rigid/affine initial transform. Points may be given in DICOM physical coordinates (`landmarkCoordSys='dicom'`, LPS/mm) or pyCERR virtual coordinates (`'cerr'`, x/y/z in cm).

```python
import numpy as np
baseLandmarksM = np.array([[x1, y1, z1], [x2, y2, z2], ...])  # fixed scan
movLandmarksM  = np.array([[x1, y1, z1], [x2, y2, z2], ...])  # moving scan

planC = ants_reg.registerScansAnts(
    planC, baseScanNum, planC, movScanNum,
    transformSaveDir=transformSaveDir,
    typeOfTransform='antsRegistrationSyNQuick[b]',
    baseMask3M=baseMask3M, movMask3M=movMask3M,
    baseLandmarksM=baseLandmarksM, movLandmarksM=movLandmarksM,
    landmarkCoordSys='cerr', landmarkTransformType='rigid')
```

## Warp the ROI into PET/CT space

`warpStructuresAnts` applies the stored transform to the moving ROI and adds the warped structure to `planC`, associated with the fixed scan.

In [ ]:
planC = ants_reg.warpStructuresAnts(planC, baseScanNum, planC,
                                    [gtvStrNum], deformS)
print('Warped structure:', planC.structure[-1].structureName,
      '-> scan', baseScanNum)

## Visualize the result

In [ ]:
# Quick visual check: central axial slice of fixed, moving, and
# warped-moving scans. The warped-moving scan is the last one added.
import numpy as np
import matplotlib.pyplot as plt

fixed3M = planC.scan[baseScanNum].getScanArray()
moving3M = planC.scan[movScanNum].getScanArray()
warped3M = planC.scan[-1].getScanArray()

slc = fixed3M.shape[2] // 2
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(fixed3M[:, :, slc], cmap='gray'); ax[0].set_title('Fixed')
ax[1].imshow(moving3M[:, :, min(slc, moving3M.shape[2] - 1)], cmap='gray')
ax[1].set_title('Moving (before)')
ax[2].imshow(warped3M[:, :, slc], cmap='gray')
ax[2].set_title('Warped moving')
for a in ax:
    a.axis('off')
plt.tight_layout(); plt.show()